In [1]:
import pandas as pd
import numpy as np

model_data = pd.read_csv('../data/processed/processed_data.csv')
print("Shape:", model_data.shape)
print(model_data.columns.tolist())

Shape: (96470, 20)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'delivery_days', 'is_outlier_delivery', 'is_late', 'purchase_month', 'purchase_dayofweek', 'customer_state', 'seller_state', 'same_state', 'price', 'freight_value', 'price_scaled', 'freight_value_scaled']


In [2]:
# Check for missing values in key modeling columns
print(model_data[['delivery_days', 'customer_state', 'seller_state', 'price_scaled', 'freight_value_scaled']].isnull().sum())

# Drop rows missing the target variable (can't train on unknown answers)
model_data = model_data.dropna(subset=['delivery_days']).copy()
print("\nShape after dropping missing target:", model_data.shape)

delivery_days           0
customer_state          0
seller_state            0
price_scaled            0
freight_value_scaled    0
dtype: int64

Shape after dropping missing target: (96470, 20)


In [3]:
# Check for missing values in key modeling columns
print(model_data[['delivery_days', 'customer_state', 'seller_state', 'price_scaled', 'freight_value_scaled']].isnull().sum())

# Drop rows missing the target variable (can't train on unknown answers)
model_data = model_data.dropna(subset=['delivery_days']).copy()
print("\nShape after dropping missing target:", model_data.shape)

delivery_days           0
customer_state          0
seller_state            0
price_scaled            0
freight_value_scaled    0
dtype: int64

Shape after dropping missing target: (96470, 20)


## Model Development

This notebook trains and compares three regression models to predict `delivery_days`:
1. **Linear Regression** — baseline model
2. **Random Forest Regression** — captures non-linear relationships
3. **XGBoost Regression** — advanced gradient boosting model

Models are evaluated using MAE, RMSE, and R² Score, per the project proposal.

In [4]:
from sklearn.model_selection import train_test_split

# One-hot encode categorical columns for modeling
model_ready = pd.get_dummies(model_data, columns=['customer_state', 'seller_state'], drop_first=True)

# Define features (X) and target (y)
feature_cols = ['purchase_month', 'purchase_dayofweek', 'same_state', 'price_scaled', 'freight_value_scaled'] + \
               [col for col in model_ready.columns if col.startswith('customer_state_') or col.startswith('seller_state_')]

X = model_ready[feature_cols]
y = model_ready['delivery_days']

# Split into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (77176, 52)
Testing set: (19294, 52)


### Model 1: Linear Regression (Baseline)

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Train the model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict on the test set
lr_predictions = lr_model.predict(X_test)

# Evaluate
lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print("Linear Regression Results:")
print(f"MAE:  {lr_mae:.2f} days")
print(f"RMSE: {lr_rmse:.2f} days")
print(f"R²:   {lr_r2:.4f}")

Linear Regression Results:
MAE:  5.42 days
RMSE: 8.25 days
R²:   0.2162


### Model 2: Random Forest Regression

In [6]:
from sklearn.ensemble import RandomForestRegressor

# Train the model
rf_model = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict on test set
rf_predictions = rf_model.predict(X_test)

# Evaluate
rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results:")
print(f"MAE:  {rf_mae:.2f} days")
print(f"RMSE: {rf_rmse:.2f} days")
print(f"R²:   {rf_r2:.4f}")

Random Forest Results:
MAE:  5.00 days
RMSE: 7.89 days
R²:   0.2831


### Model 3: XGBoost Regression

In [7]:
from xgboost import XGBRegressor

# Train the model
xgb_model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)

# Predict on test set
xgb_predictions = xgb_model.predict(X_test)

# Evaluate
xgb_mae = mean_absolute_error(y_test, xgb_predictions)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_predictions))
xgb_r2 = r2_score(y_test, xgb_predictions)

print("XGBoost Results:")
print(f"MAE:  {xgb_mae:.2f} days")
print(f"RMSE: {xgb_rmse:.2f} days")
print(f"R²:   {xgb_r2:.4f}")

XGBoost Results:
MAE:  4.92 days
RMSE: 7.84 days
R²:   0.2909


### Model Comparison

In [8]:
comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'MAE (days)': [lr_mae, rf_mae, xgb_mae],
    'RMSE (days)': [lr_rmse, rf_rmse, xgb_rmse],
    'R² Score': [lr_r2, rf_r2, xgb_r2]
})
print(comparison.to_string(index=False))

            Model  MAE (days)  RMSE (days)  R² Score
Linear Regression    5.417827     8.246809  0.216206
    Random Forest    5.004746     7.886921  0.283122
          XGBoost    4.919522     7.844076  0.290890


### Model Improvement: Adding Product Category

EDA showed product category (e.g. office_furniture) has a meaningful effect on delivery time, but it was missing from the feature set above. Let's merge it in and retrain XGBoost to see if it improves performance.

In [9]:
# Load order items + products to bring in category information
items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

# Get one category per order (first item's category, to keep one row per order)
order_category = items.merge(products, on='product_id') \
                        .merge(category_translation, on='product_category_name', how='left') \
                        .drop_duplicates(subset='order_id')[['order_id', 'product_category_name_english']]

# Merge into model_data
model_data_v2 = model_data.merge(order_category, on='order_id', how='left')
model_data_v2['product_category_name_english'] = model_data_v2['product_category_name_english'].fillna('unknown')

print("Shape after merge:", model_data_v2.shape)
print("Missing categories:", model_data_v2['product_category_name_english'].isnull().sum())

Shape after merge: (96470, 21)
Missing categories: 0


In [10]:
from sklearn.preprocessing import LabelEncoder

# Label-encode product category (many categories -> label encoding keeps feature count manageable)
le_category = LabelEncoder()
model_data_v2['category_encoded'] = le_category.fit_transform(model_data_v2['product_category_name_english'])

# One-hot encode states again (same as before)
model_ready_v2 = pd.get_dummies(model_data_v2, columns=['customer_state', 'seller_state'], drop_first=True)

# Updated feature list: original features + category_encoded
feature_cols_v2 = ['purchase_month', 'purchase_dayofweek', 'same_state', 'price_scaled', 'freight_value_scaled', 'category_encoded'] + \
                   [col for col in model_ready_v2.columns if col.startswith('customer_state_') or col.startswith('seller_state_')]

X_v2 = model_ready_v2[feature_cols_v2]
y_v2 = model_ready_v2['delivery_days']

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(X_v2, y_v2, test_size=0.2, random_state=42)

print("Training set:", X_train_v2.shape)
print("Testing set:", X_test_v2.shape)

Training set: (77176, 53)
Testing set: (19294, 53)


In [11]:
xgb_model_v2 = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model_v2.fit(X_train_v2, y_train_v2)

xgb_predictions_v2 = xgb_model_v2.predict(X_test_v2)

xgb_mae_v2 = mean_absolute_error(y_test_v2, xgb_predictions_v2)
xgb_rmse_v2 = np.sqrt(mean_squared_error(y_test_v2, xgb_predictions_v2))
xgb_r2_v2 = r2_score(y_test_v2, xgb_predictions_v2)

print("XGBoost v2 (with product category) Results:")
print(f"MAE:  {xgb_mae_v2:.2f} days")
print(f"RMSE: {xgb_rmse_v2:.2f} days")
print(f"R²:   {xgb_r2_v2:.4f}")

XGBoost v2 (with product category) Results:
MAE:  4.88 days
RMSE: 7.78 days
R²:   0.3021


In [ ]:
xgb_model_v2 = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model_v2.fit(X_train_v2, y_train_v2)

xgb_predictions_v2 = xgb_model_v2.predict(X_test_v2)

xgb_mae_v2 = mean_absolute_error(y_test_v2, xgb_predictions_v2)
xgb_rmse_v2 = np.sqrt(mean_squared_error(y_test_v2, xgb_predictions_v2))
xgb_r2_v2 = r2_score(y_test_v2, xgb_predictions_v2)

print("XGBoost v2 (with product category) Results:")
print(f"MAE:  {xgb_mae_v2:.2f} days")
print(f"RMSE: {xgb_rmse_v2:.2f} days")
print(f"R²:   {xgb_r2_v2:.4f}")

In [13]:
xgb_model_v2 = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_model_v2.fit(X_train_v2, y_train_v2)

xgb_predictions_v2 = xgb_model_v2.predict(X_test_v2)

xgb_mae_v2 = mean_absolute_error(y_test_v2, xgb_predictions_v2)
xgb_rmse_v2 = np.sqrt(mean_squared_error(y_test_v2, xgb_predictions_v2))
xgb_r2_v2 = r2_score(y_test_v2, xgb_predictions_v2)

print("XGBoost v2 (with product category) Results:")
print(f"MAE:  {xgb_mae_v2:.2f} days")
print(f"RMSE: {xgb_rmse_v2:.2f} days")
print(f"R²:   {xgb_r2_v2:.4f}")

XGBoost v2 (with product category) Results:
MAE:  4.88 days
RMSE: 7.78 days
R²:   0.3021


### Model Development — Summary

XGBoost (with product category added) is the best-performing model, though all three regression models explain only 22–30% of delivery time variance. This is expected given weak linear correlations found in earlier EDA — delivery delays are driven more by categorical/geographic factors than by the numeric features available in this dataset. Feature importance analysis (next section) will identify which specific factors matter most.

In [14]:
importance_df = pd.DataFrame({
    'Feature': feature_cols_v2,
    'Importance': xgb_model_v2.feature_importances_
}).sort_values('Importance', ascending=False)

print(importance_df.head(15))

              Feature  Importance
2          same_state    0.501837
15  customer_state_MG    0.069403
30  customer_state_SP    0.066802
22  customer_state_PR    0.037000
18  customer_state_PA    0.029271
9   customer_state_BA    0.020427
10  customer_state_CE    0.018551
6   customer_state_AL    0.018286
14  customer_state_MA    0.016189
11  customer_state_DF    0.015297
0      purchase_month    0.013190
8   customer_state_AP    0.012347
7   customer_state_AM    0.011686
52    seller_state_SP    0.008479
23  customer_state_RJ    0.008477
